# Covenant verification from bespoke credit agreements — prototype

**Scope:** Total Leverage Ratio, US (New York law style) agreements. **All agreements and financial figures in this notebook are SYNTHETIC**, written for this prototype; the cap structure mirrors patterns observed in public filings (see brief).

Pipeline: **retrieve (code) → extract (LLM) → validate (code) → approve (human) → calculate & reconcile (code) → review (human)**.
The LLM only drafts the rule spec. It never computes a number.

In [1]:
import json, os, subprocess, sys
from pathlib import Path
import pandas as pd
from covenant.parser import parse, dependencies
from covenant.schema import CovenantSpec, QuarterPackage
from covenant.validate import validate
from covenant.calculator import calculate, cap_amount
from covenant import extract as ex
pd.set_option("display.max_colwidth", 120)

## 1. Deterministic retrieval
Definitions are located by name (quoted term at the start of a paragraph in Section 1.01), not by semantic search. Dependencies are matched longest-first with masking; boilerplate and procedural terms are separated from terms that change the arithmetic.

In [2]:
agr = parse(Path("data/synthetic_agreement_v1.html").read_text())
amd = parse(Path("data/synthetic_amendment_no1.html").read_text(), require_definitions=False)
print(f"{len(agr.definitions)} definitions; sections {list(agr.sections)}; warnings {agr.warnings}")
deps = dependencies("Consolidated EBITDA", agr.definitions)
deps

18 definitions; sections ['1.01', '1.02', '7.11', '7.12']; warnings []


{'computational': ['Consolidated Interest Expense',
  'Consolidated Net Income',
  'Test Period'],
 'procedural': ['Compliance Certificate', 'Responsible Officer'],
 'stop_listed': ['Borrower'],
 'section_refs': []}

In [3]:
print(agr.definitions["Consolidated EBITDA"])

“Consolidated EBITDA” means, for any Test Period, Consolidated Net Income for such Test Period, plus, without duplication and to the extent deducted in arriving at Consolidated Net Income for such Test Period: (a) Consolidated Interest Expense; (b) provision for taxes based on income, profits or capital; (c) depreciation and amortization expense; (d) non-cash charges, excluding any non-cash charge that represents an accrual of or reserve for cash payments in any future period; (e) restructuring, integration and severance costs; (f) the amount of run-rate cost savings and synergies projected by the Borrower in good faith to be realized within 24 months after the date of the related action, net of the amount of actual benefits realized, provided that such cost savings and synergies are reasonably identifiable and factually supportable and are certified by a Responsible Officer in the Compliance Certificate; (g) adjustments in respect of new contracts and increased pricing or volume under

## 2. Gold spec (hand-built answer key) and validation
The gold spec is my own reading of the definition, encoded as data. Interpretations the text leaves open are written down, not hidden in code.

In [4]:
v1 = CovenantSpec(**json.loads(Path("data/gold_spec_v1.json").read_text()))
v2 = CovenantSpec(**json.loads(Path("data/gold_spec_v2.json").read_text()))
print("v1 validation errors:", validate(v1, agr.full_text))
print("v2 validation errors:", validate(v2, agr.full_text + " " + amd.full_text))
pd.DataFrame([{"group": g.id, "pct": g.pct, "basis": g.basis, "order": g.order, "members": g.members} for g in v1.cap_groups])

v1 validation errors: []
v2 validation errors: []


,group,pct,basis,order,members
0,G25,0.25,before_addbacks,1,"[(e), (f)]"
1,G10_PRE,0.10,before_addbacks,2,[(g)]
2,G10_POST,0.10,after_addbacks,3,[(h)]


In [5]:
for i in v1.interpretations: print("-", i)

- Caps are applied in order G25, G10_PRE, G10_POST. 'Before giving effect to such add-backs' excludes only that group; earlier groups' allowed amounts are included in later bases.
- G10_POST is circular: A <= 0.10 x (B + A), so A <= B / 9.


## 3. LLM extraction (the only probabilistic step)
Runs when `OPENROUTER_API_KEY` is set, or loads a cached run from `outputs/`. Scored field by field against the gold spec.

In [7]:
MODEL = os.environ.get("OPENROUTER_MODEL", "meta-llama/llama-3.3-70b-instruct:free")
term = "Consolidated EBITDA"
print(ex.build_prompt(term, agr.definitions[term], deps)[:900], "...")
run = ex.extract(term, agr.definitions[term], deps, MODEL)
if run is None:
    print("\nNOT RUN: set OPENROUTER_API_KEY (free key at openrouter.ai) and re-run this cell.")
elif "parse_error" in run:
    print("Model returned malformed JSON:", run["parse_error"]); print(run["raw"][:1500])
else:
    gold = json.loads(Path("data/gold_spec_v1.json").read_text())
    rows = ex.score(run["spec"], gold, agr.full_text)
    df = pd.DataFrame(rows); print(f"model: {run['model']}  fields correct: {df.ok.sum()}/{len(df)}")
    pred_errors = []
    display(df)

You convert one credit agreement definition into a JSON rule spec. You do not compute anything.

Rules:
- Use ONLY the text provided. Every "quote" must be copied VERBATIM from it (a short exact span).
- If something affects the calculation but you cannot pin it down from the text, add a sentence to "unresolved". Never guess.
- Cap "basis": "before_addbacks" or "after_addbacks", exactly as the proviso says.
- Output JSON only, no prose, no markdown fences.

Schema:
{"adjustments": [{"clause": "(a)", "label": str, "direction": "add"|"deduct", "cap_group": str|null,
   "conditions": [{"type": "max_realization_months"|"officer_certification"|"analyst_judgment", "value": number|null,
                   "description": str, "citation": {"section": str, "quote": str}}],
   "citation": {"section": str, "quote": str}}],
 "cap_groups": [{"id": str, "pct": number, "basis": str, "order": int, "membe ...
model: nvidia/nemotron-3-ultra-550b-a55b:free  fields correct: 30/31


,field,ok,note
0,adjustment (a) found,True,NaN
1,adjustment (a) direction,True,NaN
2,adjustment (b) found,True,NaN
3,adjustment (b) direction,True,NaN
4,adjustment (c) found,True,NaN
5,adjustment (c) direction,True,NaN
6,adjustment (d) found,True,NaN
7,adjustment (d) direction,True,NaN
8,adjustment (e) found,True,NaN
9,adjustment (e) direction,True,NaN


In [9]:
import json
from pathlib import Path

Path("outputs").mkdir(exist_ok=True)

Path("outputs/llm_extraction_review.json").write_text(
    json.dumps(run, indent=2, default=str)
)

print("Extraction saved.")

Extraction saved.


## 4. One quarter, fully traced: the circular post-add-back cap
The estimates-at-completion add-back (clause (h)) is capped at 10% of EBITDA *after* giving effect to it: A ≤ 0.10 × (B + A), so A ≤ B / 9. The borrower applied 10% of B.

In [ ]:
scen = json.loads(Path("data/scenarios/scenarios.json").read_text())
r = calculate([v1, v2], QuarterPackage(**scen[2]))
print(f"status {r.status} | spec v{r.spec_version} | EBITDA {r.ebitda:,.2f} | debt {r.total_debt:,.0f} | ratio {r.ratio:.3f}x vs max {r.max_ratio:.2f}x | headroom {r.headroom:.3f}x")
print("flags:", *r.flags, sep="\n  "); print("variances vs certificate:", *r.variances, sep="\n  ")
pd.DataFrame(r.lines)

## 5. All synthetic scenarios

In [ ]:
out = []
for s in scen:
    r = calculate([v1, v2], QuarterPackage(**s))
    e = s["expected"]
    out.append({"scenario": s["scenario"], "expected": e["status"], "got": r.status,
                "EBITDA": round(r.ebitda, 2), "ratio": round(r.ratio, 3), "max": r.max_ratio,
                "match": r.status == e["status"] and abs(r.ebitda - e["ebitda"]) < 0.01,
                "first flag / variance": (r.flags + r.variances + r.review_items + [""])[0]})
res = pd.DataFrame(out); print(f"{res.match.sum()}/{len(res)} scenarios match hand-computed expectations"); res

## 6. Full test suite (parser, validator, calculator, known limitation)

In [ ]:
print(subprocess.run([sys.executable, "-m", "pytest", "-q", "-rx", "--color=no", "-p", "no:cacheprovider", "tests"], capture_output=True, text=True).stdout)

## 7. Where it fails (honest report)

1. **Validator cannot catch a wrong interpretation with a correct quote** (test V4, expected failure). If the LLM cites the right proviso but sets `basis` to the wrong value, every quote still validates. Only the analyst's approval step catches this. Mitigation idea: a second, independent extraction pass, with disagreements routed to the analyst.
2. **Parser not yet run on a full EDGAR filing.** It is tested on a synthetic agreement that includes a table of contents, page footers, multi-paragraph definitions and plurals. Real filings will have formatting this does not cover; the parser fails loudly rather than guessing (see section 8).
3. **Synthetic agreement is simpler than real ones.** No pro forma section (e.g. Section 1.11 style), no contractual historical EBITDA overrides, and `Indebtedness` is referenced to Section 7.03 without being resolved.
4. **Cap evaluation order is an interpretation.** Later groups' bases include earlier groups' allowed amounts. Another reading is defensible; the spec records the choice so an analyst approves it explicitly.
5. **"Good faith" and "factually supportable" are not checked.** The system can only route synergy add-backs to review; it cannot judge whether savings are real.
6. **Tolerance hides small errors.** Ratios are compared at ±0.005x so that borrower rounding does not raise noise; a genuine 0.004x error would pass.

## 8. Optional: run the parser on a real EDGAR exhibit
Save a credit agreement exhibit (HTML) into `data/real/` and run this cell. If SEC blocks a scripted download, send a `User-Agent: Your Name email` header or save the page from a browser.

In [ ]:
for f in sorted(Path("data/real").glob("*.htm*")) if Path("data/real").exists() else []:
    try:
        a = parse(f.read_text(encoding="utf-8", errors="replace"))
        key = next((k for k in a.definitions if k.casefold() == "consolidated ebitda"), None)
        print(f.name, "| definitions:", len(a.definitions), "| warnings:", a.warnings[:5])
        if key: print(dependencies(key, a.definitions)); print(a.definitions[key][:1500])
    except ValueError as e:
        print(f.name, "FAILED LOUDLY:", e)

In [ ]:
print("Model being used:", MODEL)